# 04 -- Layer 2: Adaptive Baseline + Surgical Graph Inspection

Causal (R2-3 fix) surge detection + persistence/co-occurrence/burst targeting
(`src/awik/adaptive_baseline.py`), then the G1/G2/G3/G5/G6 graph queries
(`src/awik/graph_inspection.py`, G4 permanently disabled). Also runs the
routing ablation (Cell 7b/7c in the original): what happens to recall if the
adaptive-baseline routing is disabled and graph inspection only sees
`l1_flagged`.

**Determinism fix:** `src/awik/graph_inspection.py`'s G1/G2/G3/G4(code-G5) queries previously carried a per-batch `LIMIT` (200/200/100/100/300) that bound in nearly every batch at this pipeline's population sizes, truncating match counts and making the union depend on `PYTHONHASHSEED` through batch composition. The LIMIT clauses are removed and candidate lists below are built with `sorted()`, not a bare `list(some_set)`, so results no longer depend on Python's set-iteration order.

**Causal PC ownership (R2-3-style fix, now applied to G5(paper)/code-G6):** `Q_OWNER` previously computed majority PC ownership once, globally, over the entire dataset -- a non-causal design. It now takes `$year`/`$week` parameters and is queried once per distinct ISO week present in `surge_pairs` (via `fetch_owner_of_by_week`), using only logons strictly before that week's start. `run_g6` looks up each row's owner from its own (Year, Week), never from the tested week itself or later.

In [1]:
import os
import sys
sys.path.insert(0, '../src')
import pandas as pd

from awik.neo4j_client import Neo4jConnection
from awik.adaptive_baseline import compute_surge_signals, target_users
from awik import graph_inspection as gi
from awik import io

df_master = io.load_df('01_df_master')
weeks_df = io.load_df('01_weeks_df')
state03 = io.load_json('03_layer1_state')
l1_flagged = set(state03['l1_flagged'])

db_conn = Neo4jConnection("bolt://localhost:7687", "neo4j", os.environ.get("NEO4J_PASSWORD", "changeme"))

=> Neo4j: connected.


## Adaptive baseline (causal) + targeting

In [2]:
print("-> Adaptive Baseline v4 -- causal fix for R2-3 (expanding, trailing-only stats)...")
df_sorted, df_surges = compute_surge_signals(df_master)
print(f"[CAUSAL] total surges detected: {len(df_surges):,}")

targeting = target_users(df_surges, n_weeks=len(weeks_df))
users_targeted = targeting['users_targeted']
users_surged = targeting['users_surged']
per_user = targeting['per_user']

candidates_graph = sorted(users_targeted & l1_flagged)
near_miss_group = sorted(users_targeted - l1_flagged)

def surged(col):
    return set(df_surges.loc[df_surges[col]==1, 'User_ID']) if not df_surges.empty else set()

email_near_miss = sorted(surged('lon_email') & set(near_miss_group))

pop = df_master['User_ID'].nunique()
print(f"K_persist={targeting['k_persist']}, K_cooccur={targeting['k_cooccur']}, K_burst={targeting['k_burst']}")
print(f"users_targeted (surge-targeted pool T): {len(users_targeted)} ({len(users_targeted)/pop*100:.1f}% population)")
print(f"candidates_graph: {len(candidates_graph):,} | near_miss_group: {len(near_miss_group):,} "
      f"| email_near_miss: {len(email_near_miss):,}")
print(f"users_surged (surge population, feeds G6): {len(users_surged):,} ({len(users_surged)/pop*100:.1f}%)")

-> Adaptive Baseline v4 -- causal fix for R2-3 (expanding, trailing-only stats)...


[CAUSAL] total surges detected: 15,355
K_persist=11, K_cooccur=2, K_burst=3
users_targeted (surge-targeted pool T): 160 (4.0% population)
candidates_graph: 132 | near_miss_group: 28 | email_near_miss: 27
users_surged (surge population, feeds G6): 3,607 (90.2%)


## Graph inspection (G1, G2, G3, G5, G6) -- original routing

In [3]:
print("=" * 65)
print("[Layer 2] Surgical Contextual Graph Inspection")
print("=" * 65)

EMAIL_P99 = gi.fetch_email_p99(db_conn)
print(f"Email size p99 (label-free): {EMAIL_P99:,.0f}")
q_g5 = gi.build_q_g5(EMAIL_P99)

surge_pairs = (df_surges[['User_ID','Year','Week']].drop_duplicates()
               .apply(lambda r: {'uid': r['User_ID'], 'year': int(r['Year']), 'week': int(r['Week'])}, axis=1)
               .tolist())

owner_of_by_week = gi.fetch_owner_of_by_week(db_conn, [(p['year'], p['week']) for p in surge_pairs])
print(f"Causal PC ownership computed for {len(owner_of_by_week)} distinct ISO weeks "
      f"(each using only logons strictly before that week's start)")

df_g1, df_g2, df_g3a, df_g3b, df_g5 = gi.run_g1_g5(db_conn, candidates_graph, email_near_miss, q_g5)

df_g6 = gi.run_g6(db_conn, surge_pairs, owner_of_by_week)

summary = gi.summarize(df_g1, df_g2, df_g3a, df_g3b, df_g5, df_g6)
g1_suspects, g2_suspects, g3_suspects = summary['g1'], summary['g2'], summary['g3']
g5_suspects, g6_suspects = summary['g5'], summary['g6']
graph_confirmed = summary['graph_confirmed']
df_forensic = summary['df_forensic']
df_g3_vis = summary['df_g3_vis']

print(f"G1 After-Hours Lateral   : {len(g1_suspects):4d} users ({len(df_g1):,} events)")
print(f"G2 Lateral -> Mass Email : {len(g2_suspects):4d} users ({len(df_g2):,} events)")
print(f"G3 Lateral+LargeEmail    : {len(g3_suspects):4d} users")
print(f"G4 Exfil Watchlist       : disabled permanently (VISITED_URL removed)")
print(f"G5 Mass Email Near-Miss  : {len(g5_suspects):4d} users ({len(df_g5):,} events)")
print(f"G6 Attacker Lateral (AH) : {len(g6_suspects):4d} users ({len(df_g6):,} events)")
print(f"Total graph confirmed    : {len(graph_confirmed):4d} unique users")

[Layer 2] Surgical Contextual Graph Inspection


Email size p99 (label-free): 5,377,734


Causal PC ownership computed for 74 distinct ISO weeks (each using only logons strictly before that week's start)


G1 After-Hours Lateral   :  132 users (38,875 events)
G2 Lateral -> Mass Email :  132 users (87,661 events)
G3 Lateral+LargeEmail    :  128 users
G4 Exfil Watchlist       : disabled permanently (VISITED_URL removed)
G5 Mass Email Near-Miss  :   21 users (229 events)
G6 Attacker Lateral (AH) :  178 users (14,781 events)
Total graph confirmed    :  300 unique users


## Routing ablation [PERMANENT]: same graph inspection, but fed only `l1_flagged`

Tests whether persistence/co-occurrence/burst targeting is load-bearing, or graph
inspection would catch the same actors from Layer-1-flagged accounts alone.

In [4]:
print("=" * 65)
print("[Ablation] Layer 2 WITHOUT Adaptive-Baseline Routing")
print("=" * 65)

candidates_graph_ablasi = sorted(l1_flagged)
email_near_miss_ablasi = []
surge_pairs_ablasi = (
    df_surges[df_surges['User_ID'].isin(l1_flagged)][['User_ID', 'Year', 'Week']].drop_duplicates()
    .apply(lambda r: {'uid': r['User_ID'], 'year': int(r['Year']), 'week': int(r['Week'])}, axis=1)
    .tolist()
)
print(f"Ablation population: candidates_graph_ablasi={len(candidates_graph_ablasi)} (= l1_flagged, near-miss disabled)")

df_g1_a, df_g2_a, df_g3a_a, df_g3b_a, df_g5_a = gi.run_g1_g5(
    db_conn, candidates_graph_ablasi, email_near_miss_ablasi, q_g5)
df_g6_a = gi.run_g6(db_conn, surge_pairs_ablasi, owner_of_by_week)

summary_a = gi.summarize(df_g1_a, df_g2_a, df_g3a_a, df_g3b_a, df_g5_a, df_g6_a)
graph_confirmed_ablasi = summary_a['graph_confirmed']
print(f"G1_a={len(summary_a['g1'])} G2_a={len(summary_a['g2'])} G3_a={len(summary_a['g3'])} "
      f"G5_a={len(summary_a['g5'])} G6_a={len(summary_a['g6'])}")
print(f"Total graph_confirmed_ablasi: {len(graph_confirmed_ablasi)} unique users")

[Ablation] Layer 2 WITHOUT Adaptive-Baseline Routing
Ablation population: candidates_graph_ablasi=1056 (= l1_flagged, near-miss disabled)


G1_a=620 G2_a=1056 G3_a=1000 G5_a=0 G6_a=69
Total graph_confirmed_ablasi: 1056 unique users


In [5]:
io.save_df(df_surges, '04_df_surges')
io.save_df(df_forensic, '04_df_forensic')
io.save_df(df_g1, '04_df_g1'); io.save_df(df_g2, '04_df_g2')
io.save_df(df_g3a, '04_df_g3a'); io.save_df(df_g3_vis, '04_df_g3_vis')
io.save_df(df_g6, '04_df_g6')

io.save_json({
    'candidates_graph': candidates_graph, 'near_miss_group': near_miss_group,
    'email_near_miss': email_near_miss, 'users_targeted': sorted(users_targeted),
    'users_surged': sorted(users_surged),
    'g1_suspects': sorted(g1_suspects), 'g2_suspects': sorted(g2_suspects),
    'g3_suspects': sorted(g3_suspects), 'g5_suspects': sorted(g5_suspects),
    'g6_suspects': sorted(g6_suspects), 'graph_confirmed': sorted(graph_confirmed),
    'candidates_graph_ablasi': candidates_graph_ablasi,
    'graph_confirmed_ablasi': sorted(graph_confirmed_ablasi),
    'g1_suspects_a': sorted(summary_a['g1']), 'g2_suspects_a': sorted(summary_a['g2']),
    'g3_suspects_a': sorted(summary_a['g3']), 'g5_suspects_a': sorted(summary_a['g5']),
    'g6_suspects_a': sorted(summary_a['g6']),
    'k_persist': targeting['k_persist'], 'k_cooccur': targeting['k_cooccur'], 'k_burst': targeting['k_burst'],
}, '04_layer2_state')
db_conn.close()
print("Checkpoint saved -> data/interim/04_*")

Checkpoint saved -> data/interim/04_*
